In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from microimage import Reader
from pathlib import Path
import xarray as xr
filename = Path("/media/cephfs/sankar/Images for Jerome/180925_GFP 488_PCNT 555_new_SunBicD2_siG2BP1_multiple_2.lsm")

with Reader(filename) as f:
#    print(f.metadata.keys())
#    print(f.metadata["ChannelColors"]["ColorNames"])
#    print(f.metadata)
   for s in f.scenes():
      print(s)
      break

(6, 3, 1584, 1584)
scene coords {'Z': [0.0, 6.336906372008041e-07, 1.2673812744016082e-06, 1.9010719116024121e-06, 2.5347625488032163e-06, 3.1684531860040205e-06], 'C': ['ChS1-T1', 'Ch1-T2', 'Ch2-T2'], 'Y': [0.0, 8.519642863826472e-08, 1.7039285727652944e-07, 2.5558928591479413e-07, 3.4078571455305887e-07, 4.259821431913236e-07, 5.111785718295883e-07, 5.96375000467853e-07, 6.815714291061177e-07, 7.667678577443825e-07, 8.519642863826472e-07, 9.371607150209119e-07, 1.0223571436591765e-06, 1.1075535722974413e-06, 1.192750000935706e-06, 1.2779464295739707e-06, 1.3631428582122355e-06, 1.4483392868505002e-06, 1.533535715488765e-06, 1.6187321441270297e-06, 1.7039285727652945e-06, 1.789125001403559e-06, 1.8743214300418238e-06, 1.9595178586800885e-06, 2.044714287318353e-06, 2.129910715956618e-06, 2.2151071445948825e-06, 2.3003035732331475e-06, 2.385500001871412e-06, 2.470696430509677e-06, 2.5558928591479415e-06, 2.6410892877862065e-06, 2.726285716424471e-06, 2.8114821450627355e-06, 2.8966785737

In [ ]:
from microimage import ExperimentManager, imreader
# from dask.distributed import Client, LocalCluster, Future
# from functools import partial

class MetadataParser:
    def __init__(self, extra_metadata=None):
        self.extra_metadata = extra_metadata
    def __call__(self,path):
        items = []
        with imreader(path) as img: # open the file
            
            for k, scene in enumerate(img.scenes()):
                  items.append({
                      "internal_index": k,
                      "metadata": scene.metadata()
                  })
        return items

# p = parse_metadata({"dapi": 1, "nuclei": 2, "pcnt": 0,"condition":"wt"})


data_dir = "/media/cephfs/sankar/Images for Jerome"
results_dir = "./results"
db_path = "project.db"
m = ExperimentManager(data_dir, results_dir, db_path)
p = MetadataParser({"dapi": 1, "nuclei": 2, "pcnt": 0})
m.scan_data_directory(["lsm"], p)

Project root (Base for result paths) set to: /home/jeromeb/work/code/microimage
Initializing database at /home/jeromeb/work/code/microimage/project.db...
Database structure verified/created.
Scanning /media/cephfs/sankar/Images for Jerome for new files...
  Skipping hidden resource file: ._180925_GFP 488_PCNT 555_new_SunBicD2_siG2BP1_multiple_2.lsm
  Using parser to analyze file: 180925_GFP 488_PCNT 555_new_SunBicD2_siG2BP1_multiple_2.lsm
File Parser Error during scan: 'dict' object is not callable


In [ ]:
import numpy as np
src = np.zeros((10,512,512,5,2))
src_axes = "PXYZC" 
P = 0
dst_axes = "TCZYX"

# source indices
T1 = [k for k,e in enumerate(dst_axes) if src_axes.find(e)!=-1]
# destination indices
T2 = [src_axes.find(e) for k,e in enumerate(dst_axes) if src_axes.find(e)!=-1]
# extract axes
[slice(None)]* src.ndims()
D = [k for k,e in enumerate(src_axes) if dst_axes.find(e)==-1]
# new axes
A = [k for k,e in enumerate(src_axes) if dst_axes.find(e)==-1]
print(src_axes,dst_axes,T1,T2,A,D)


dst = np.moveaxis(src, T1, T2)


dst = np.expand_dims(dst, A)
src.shape, dst.shape

PXYZC TCZYX [1, 2, 3, 4] [4, 3, 2, 1] [0] [0]


((10, 512, 512, 5, 2), (1, 10, 2, 5, 512, 512))

In [26]:
from ome_types import OME
from ome_types.model import Image

Image(description="test")

ValidationError: 1 validation error for Image
pixels
  Field required [type=missing, input_value={'description': 'test'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing

In [31]:
from ome_types.model import Channel
from ome_types.model import Image
from ome_types.model import OME
from ome_types.model import Pixels
from ome_types.model import TiffData
import uuid

ome = OME(uuid=f"urn:uuid:{uuid.uuid4()}")
pixels = Pixels(
    dimension_order='XYZCT',
    physical_size_x="0.2516",
    physical_size_y="0.2516",
    physical_size_z="0.75",
    size_x=256,
    size_y=256,
    size_z=31,
    size_c=3,
    size_t=40,
    type='uint16')
pixels.channels.extend([
  Channel(color="16711935", name="NCAPD2", samples_per_pixel=1),
  Channel(color="65535", name="DNA", samples_per_pixel=1),
  Channel(color="-1", name="NEG_Dextran", samples_per_pixel=1)])
ome.images.append(Image(name="cell0005_R0001", pixels=pixels))

ome

OME(
   images=[<1 field_type>],
   uuid='urn:uuid:5984f1c5-37d5-49dd-a375-57223629e0d7',
)

In [ ]:
import xarray as xr
data = xr.DataArray([[10,10,10],[10,10,10]], dims=("y","x"))
data.attrs["long_name"] 

<xarray.DataArray (y: 2, x: 3)> Size: 48B
array([[10, 10, 10],
       [10, 10, 10]])
Dimensions without coordinates: y, x